In [2]:

import pandas as pd
from pathlib import Path

# Project root
PROJECT_ROOT = Path.cwd().parent

print("Project root:", PROJECT_ROOT)

# Existing curated dataset directory
curated_dir = PROJECT_ROOT / "data" / "processed" / "curated"

# Load the four existing datasets
datasets = {
    "dishes": pd.read_csv(curated_dir / "dishes.csv"),
    "ingredients": pd.read_csv(curated_dir / "ingredients.csv"),
    "ingredient_reference": pd.read_csv(curated_dir / "ingredient_reference.csv"),
    "dish_nutrition": pd.read_csv(curated_dir / "dish_nutrition.csv")
}

# Confirm successful loading
for name, df in datasets.items():
    print(f"{name}: {df.shape}")


Project root: C:\Users\dell\NutriVision
dishes: (10, 3)
ingredients: (35, 3)
ingredient_reference: (9, 3)
dish_nutrition: (10, 8)


In [3]:

import pandas as pd
from pathlib import Path

# Project root
PROJECT_ROOT = Path.cwd().parent

# Output directory
text_dir = PROJECT_ROOT / "data" / "processed" / "text"
text_dir.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# 1. Reference food descriptions
# --------------------------------------------------

food_descriptions = [
    {
        "dish_id": "DISH001",
        "description": "A rice dish containing chicken breast, white rice, grape tomatoes and yellow onions."
    },
    {
        "dish_id": "DISH002",
        "description": "A rice dish made with ground beef, white rice, grape tomatoes and yellow onions."
    },
    {
        "dish_id": "DISH003",
        "description": "A rice-based meal containing Atlantic salmon, white rice, grape tomatoes and yellow onions."
    },
    {
        "dish_id": "DISH004",
        "description": "A vegetable omelette containing whole eggs, grape tomatoes and yellow onions."
    },
    {
        "dish_id": "DISH005",
        "description": "A pasta dish combining chicken breast, spaghetti and grape tomatoes."
    },
    {
        "dish_id": "DISH006",
        "description": "Spaghetti served with ground beef and grape tomatoes."
    },
    {
        "dish_id": "DISH007",
        "description": "A potato-based dish containing russet potatoes, chicken breast and yellow onions."
    },
    {
        "dish_id": "DISH008",
        "description": "An egg-based dish combining whole eggs, tomatoes and yellow onions."
    },
    {
        "dish_id": "DISH009",
        "description": "A mixed bowl containing chicken breast, russet potatoes, grape tomatoes and yellow onions."
    },
    {
        "dish_id": "DISH010",
        "description": "A mixed bowl made with ground beef, russet potatoes, grape tomatoes and yellow onions."
    }
]

description_df = pd.DataFrame(food_descriptions)

# Add dish names and categories from the existing dataset
description_df = (
    datasets["dishes"][["dish_id", "dish_name", "category"]]
    .merge(description_df, on="dish_id", how="left")
)

# Add ingredient names from the existing ingredient reference
ingredient_names = {
    "ING001": "grape tomatoes",
    "ING002": "whole eggs",
    "ING003": "yellow onions",
    "ING004": "russet potatoes",
    "ING005": "white rice",
    "ING006": "ground beef",
    "ING007": "chicken breast",
    "ING008": "Atlantic salmon",
    "ING009": "spaghetti"
}

ingredient_details = datasets["ingredients"].copy()
ingredient_details["ingredient_name"] = (
    ingredient_details["ingredient_id"].map(ingredient_names)
)

ingredients_text = (
    ingredient_details.groupby("dish_id")["ingredient_name"]
    .apply(lambda x: ", ".join(x))
    .reset_index(name="ingredients_text")
)

description_df = description_df.merge(
    ingredients_text,
    on="dish_id",
    how="left"
)

description_df = description_df[
    ["dish_id", "dish_name", "category",
     "ingredients_text", "description"]
]

# --------------------------------------------------
# 2. Independent retrieval queries
# --------------------------------------------------

retrieval_queries = [
    ("Q001", "Chicken breast accompanied by white rice with tomato and onion.", "DISH001"),
    ("Q002", "A meal of rice with chicken, grape tomatoes and yellow onions.", "DISH001"),

    ("Q003", "Ground beef served alongside rice, tomatoes and onion.", "DISH002"),
    ("Q004", "White rice combined with minced beef and fresh tomatoes.", "DISH002"),

    ("Q005", "Atlantic fish with rice, grape tomatoes and yellow onion.", "DISH003"),
    ("Q006", "A salmon meal accompanied by white rice and vegetables.", "DISH003"),

    ("Q007", "An omelette prepared with eggs, tomatoes and onion.", "DISH004"),
    ("Q008", "Whole eggs combined with grape tomatoes and yellow onions.", "DISH004"),

    ("Q009", "Chicken breast with spaghetti and tomatoes.", "DISH005"),
    ("Q010", "A pasta meal containing chicken and grape tomatoes.", "DISH005"),

    ("Q011", "Spaghetti with minced beef and tomato.", "DISH006"),
    ("Q012", "A pasta meal combining ground meat and grape tomatoes.", "DISH006"),

    ("Q013", "Russet potatoes accompanied by chicken and onion.", "DISH007"),
    ("Q014", "A potato dish containing chicken breast and yellow onions.", "DISH007"),

    ("Q015", "Eggs with tomatoes and onions.", "DISH008"),
    ("Q016", "A dish combining whole eggs with a generous amount of tomatoes.", "DISH008"),

    ("Q017", "A bowl of chicken, potatoes, tomatoes and onions.", "DISH009"),
    ("Q018", "Chicken breast with russet potatoes and vegetables.", "DISH009"),

    ("Q019", "Ground beef with potatoes, tomatoes and onions.", "DISH010"),
    ("Q020", "A bowl containing minced beef and russet potatoes with vegetables.", "DISH010")
]

queries_df = pd.DataFrame(
    retrieval_queries,
    columns=["query_id", "query_text", "expected_dish_id"]
)

# --------------------------------------------------
# 3. Save datasets
# --------------------------------------------------

description_path = text_dir / "food_descriptions.csv"
queries_path = text_dir / "retrieval_queries.csv"

description_df.to_csv(description_path, index=False)
queries_df.to_csv(queries_path, index=False)

print("Datasets created successfully!")
print("Food descriptions:", description_path)
print("Retrieval queries:", queries_path)

print("\nReference dataset shape:", description_df.shape)
print("Query dataset shape:", queries_df.shape)

display(description_df)
display(queries_df)

Datasets created successfully!
Food descriptions: C:\Users\dell\NutriVision\data\processed\text\food_descriptions.csv
Retrieval queries: C:\Users\dell\NutriVision\data\processed\text\retrieval_queries.csv

Reference dataset shape: (10, 5)
Query dataset shape: (20, 3)


,dish_id,dish_name,category,ingredients_text,description
0,DISH001,Chicken with Rice,Rice-based,"chicken breast, white rice, grape tomatoes, ye...","A rice dish containing chicken breast, white r..."
1,DISH002,Beef with Rice,Rice-based,"ground beef, white rice, grape tomatoes, yello...","A rice dish made with ground beef, white rice,..."
2,DISH003,Salmon with Rice,Rice-based,"Atlantic salmon, white rice, grape tomatoes, y...","A rice-based meal containing Atlantic salmon, ..."
3,DISH004,Vegetable Omelette,Egg-based,"whole eggs, grape tomatoes, yellow onions","A vegetable omelette containing whole eggs, gr..."
4,DISH005,Chicken Pasta,Pasta,"chicken breast, spaghetti, grape tomatoes","A pasta dish combining chicken breast, spaghet..."
5,DISH006,Spaghetti with Ground Beef,Pasta,"ground beef, spaghetti, grape tomatoes",Spaghetti served with ground beef and grape to...
6,DISH007,Baked Potato with Chicken,Potato-based,"russet potatoes, chicken breast, yellow onions",A potato-based dish containing russet potatoes...
7,DISH008,Tomato and Egg Dish,Egg-based,"whole eggs, grape tomatoes, yellow onions","An egg-based dish combining whole eggs, tomato..."
8,DISH009,Chicken and Vegetable Bowl,Mixed,"chicken breast, russet potatoes, grape tomatoe...","A mixed bowl containing chicken breast, russet..."
9,DISH010,Beef and Potato Bowl,Mixed,"ground beef, russet potatoes, grape tomatoes, ...","A mixed bowl made with ground beef, russet pot..."


,query_id,query_text,expected_dish_id
0,Q001,Chicken breast accompanied by white rice with ...,DISH001
1,Q002,"A meal of rice with chicken, grape tomatoes an...",DISH001
2,Q003,"Ground beef served alongside rice, tomatoes an...",DISH002
3,Q004,White rice combined with minced beef and fresh...,DISH002
4,Q005,"Atlantic fish with rice, grape tomatoes and ye...",DISH003
5,Q006,A salmon meal accompanied by white rice and ve...,DISH003
6,Q007,"An omelette prepared with eggs, tomatoes and o...",DISH004
7,Q008,Whole eggs combined with grape tomatoes and ye...,DISH004
8,Q009,Chicken breast with spaghetti and tomatoes.,DISH005
9,Q010,A pasta meal containing chicken and grape toma...,DISH005


In [4]:
# --------------------------------------------------
# Step 2.0.3 - Validate textual datasets
# --------------------------------------------------

description_df = pd.read_csv(description_path)
queries_df = pd.read_csv(queries_path)

# 1. Check unique identifiers
assert description_df["dish_id"].is_unique, "Duplicate dish IDs!"
assert queries_df["query_id"].is_unique, "Duplicate query IDs!"

# 2. Check dish references
valid_dish_ids = set(datasets["dishes"]["dish_id"])

assert set(description_df["dish_id"]) == valid_dish_ids, \
    "Reference descriptions do not match the original dishes!"

assert set(queries_df["expected_dish_id"]).issubset(valid_dish_ids), \
    "Some queries reference unknown dishes!"

# 3. Check missing and empty text
assert description_df["description"].notna().all()
assert description_df["ingredients_text"].notna().all()
assert queries_df["query_text"].notna().all()

assert description_df["description"].str.strip().ne("").all()
assert queries_df["query_text"].str.strip().ne("").all()

# 4. Check ingredient references
ingredient_ids = set(datasets["ingredient_reference"]["ingredient_id"])
used_ingredient_ids = set(datasets["ingredients"]["ingredient_id"])

assert used_ingredient_ids.issubset(ingredient_ids), \
    "Some ingredients have no reference!"

# 5. Check every dish has ingredients
ingredient_counts = datasets["ingredients"].groupby("dish_id").size()

assert set(ingredient_counts.index) == valid_dish_ids, \
    "Some dishes have no ingredients!"

# 6. Check query distribution
query_counts = queries_df.groupby("expected_dish_id").size()

assert (query_counts == 2).all(), \
    "Each dish should have exactly two test queries!"

assert len(query_counts) == len(valid_dish_ids)

# 7. Check duplicate text
assert not description_df["description"].duplicated().any()
assert not queries_df["query_text"].duplicated().any()

# 8. Display validation results
print("=" * 55)
print("TEXT DATASET VALIDATION")
print("=" * 55)

print("Reference dishes:", len(description_df))
print("Retrieval queries:", len(queries_df))
print("Unique dish IDs:", description_df["dish_id"].nunique())
print("Unique query IDs:", queries_df["query_id"].nunique())
print("Missing descriptions:", description_df["description"].isna().sum())
print("Missing ingredient lists:", description_df["ingredients_text"].isna().sum())
print("Missing queries:", queries_df["query_text"].isna().sum())
print("Invalid expected dish IDs:",
      (~queries_df["expected_dish_id"].isin(valid_dish_ids)).sum())

print("\nQueries per expected dish:")
display(query_counts.rename("query_count").to_frame())

print("\nAll validation checks passed!")

TEXT DATASET VALIDATION
Reference dishes: 10
Retrieval queries: 20
Unique dish IDs: 10
Unique query IDs: 20
Missing descriptions: 0
Missing ingredient lists: 0
Missing queries: 0
Invalid expected dish IDs: 0

Queries per expected dish:


,query_count
expected_dish_id,
DISH001,2
DISH002,2
DISH003,2
DISH004,2
DISH005,2
DISH006,2
DISH007,2
DISH008,2
DISH009,2



All validation checks passed!


## 2.1 Text Preprocessing

This section prepares the reference food descriptions and retrieval queries
for classical NLP processing.

The objective is to normalize textual data, handle punctuation and special
characters, tokenize text, examine stop words, and apply lemmatization
where appropriate.

In [5]:
# --------------------------------------------------
# Step 2.1.1 - Check NLP library availability
# --------------------------------------------------

import importlib.util

libraries = {
    "NLTK": "nltk",
    "spaCy": "spacy",
    "Scikit-learn": "sklearn"
}

for library_name, module_name in libraries.items():
    available = importlib.util.find_spec(module_name) is not None
    print(f"{library_name}: {'Available' if available else 'Not installed'}")

# Load the text datasets
description_df = pd.read_csv(description_path)
queries_df = pd.read_csv(queries_path)

print("\nReference descriptions:", len(description_df))
print("Retrieval queries:", len(queries_df))

print("\nExample reference description:")
print(description_df.loc[0, "description"])

print("\nExample retrieval query:")
print(queries_df.loc[0, "query_text"])

NLTK: Available
spaCy: Not installed
Scikit-learn: Available

Reference descriptions: 10
Retrieval queries: 20

Example reference description:
A rice dish containing chicken breast, white rice, grape tomatoes and yellow onions.

Example retrieval query:
Chicken breast accompanied by white rice with tomato and onion.


### 2.1.2 Text Normalization and Cleaning

The text is normalized to ensure consistent comparison between reference
descriptions and retrieval queries.

The following operations are applied:
- Conversion to lowercase.
- Replacement of punctuation and special characters with spaces.
- Removal of unnecessary whitespace.

Stop-word removal and lemmatization are considered separately to avoid
removing or altering meaningful food-related vocabulary.

In [6]:
# --------------------------------------------------
# Step 2.1.2 - Text normalization and cleaning
# --------------------------------------------------

import re

def clean_text(text):
    """
    Normalize and clean English food-related text.
    """
    text = str(text)

    # Convert to lowercase
    text = text.lower()

    # Replace punctuation and special characters with spaces
    text = re.sub(r"[^a-z0-9\s]", " ", text)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


# Apply cleaning to reference descriptions
description_df["clean_description"] = (
    description_df["description"].apply(clean_text)
)

# Apply cleaning to retrieval queries
queries_df["clean_query"] = (
    queries_df["query_text"].apply(clean_text)
)

# Display representative examples
print("REFERENCE DESCRIPTION")
print("Original:", description_df.loc[0, "description"])
print("Cleaned: ", description_df.loc[0, "clean_description"])

print("\nRETRIEVAL QUERY")
print("Original:", queries_df.loc[0, "query_text"])
print("Cleaned: ", queries_df.loc[0, "clean_query"])

REFERENCE DESCRIPTION
Original: A rice dish containing chicken breast, white rice, grape tomatoes and yellow onions.
Cleaned:  a rice dish containing chicken breast white rice grape tomatoes and yellow onions

RETRIEVAL QUERY
Original: Chicken breast accompanied by white rice with tomato and onion.
Cleaned:  chicken breast accompanied by white rice with tomato and onion


In [7]:
print("Description columns:")
print(description_df.columns.tolist())

print("\nQuery columns:")
print(queries_df.columns.tolist())

Description columns:
['dish_id', 'dish_name', 'category', 'ingredients_text', 'description', 'clean_description']

Query columns:
['query_id', 'query_text', 'expected_dish_id', 'clean_query']


In [8]:
# --------------------------------------------------
# Step 2.1.3 - Tokenization
# --------------------------------------------------

def tokenize_text(text):
    """
    Split cleaned text into individual tokens.
    """
    return text.split()


# Tokenize reference descriptions
description_df["tokens"] = (
    description_df["clean_description"].apply(tokenize_text)
)

# Tokenize retrieval queries
queries_df["tokens"] = (
    queries_df["clean_query"].apply(tokenize_text)
)

# Display representative examples
print("REFERENCE DESCRIPTION")
print(description_df.loc[0, "tokens"])

print("\nRETRIEVAL QUERY")
print(queries_df.loc[0, "tokens"])

REFERENCE DESCRIPTION
['a', 'rice', 'dish', 'containing', 'chicken', 'breast', 'white', 'rice', 'grape', 'tomatoes', 'and', 'yellow', 'onions']

RETRIEVAL QUERY
['chicken', 'breast', 'accompanied', 'by', 'white', 'rice', 'with', 'tomato', 'and', 'onion']


### 2.1.4 Stop-word Analysis

Stop words are common words that may contribute little to distinguishing
food descriptions.

Their occurrence is examined before deciding whether to remove them.
Food-related terms and meaningful descriptive vocabulary must be preserved.

In [9]:
# --------------------------------------------------
# Step 2.1.4 - Stop-word analysis
# --------------------------------------------------

from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# Combine tokens from both datasets
all_tokens = (
    description_df["tokens"].tolist()
    + queries_df["tokens"].tolist()
)

# Flatten token lists
all_tokens = [
    token
    for sentence in all_tokens
    for token in sentence
]

# Identify stop words present in the datasets
present_stop_words = sorted(
    set(all_tokens).intersection(ENGLISH_STOP_WORDS)
)

print("Stop words found in our datasets:")
print(present_stop_words)

print("\nNumber of stop words found:", len(present_stop_words))

Stop words found in our datasets:
['a', 'amount', 'an', 'and', 'by', 'made', 'of', 'whole', 'with']

Number of stop words found: 9


In [10]:
# Compare tokenization before and after stop-word removal

example_tokens = description_df.loc[0, "tokens"]

filtered_tokens = [
    token for token in example_tokens
    if token not in ENGLISH_STOP_WORDS
]

print("Original tokens:")
print(example_tokens)

print("\nTokens after stop-word removal:")
print(filtered_tokens)

Original tokens:
['a', 'rice', 'dish', 'containing', 'chicken', 'breast', 'white', 'rice', 'grape', 'tomatoes', 'and', 'yellow', 'onions']

Tokens after stop-word removal:
['rice', 'dish', 'containing', 'chicken', 'breast', 'white', 'rice', 'grape', 'tomatoes', 'yellow', 'onions']


In [11]:
# --------------------------------------------------
# Step 2.1.4 - Selective stop-word removal
# --------------------------------------------------

custom_stop_words = {
    "a",
    "an",
    "and",
    "by",
    "of",
    "with"
}

def remove_stop_words(tokens):
    """
    Remove selected grammatical stop words while
    preserving meaningful food-related vocabulary.
    """
    return [
        token for token in tokens
        if token not in custom_stop_words
    ]


# Apply to reference descriptions
description_df["filtered_tokens"] = (
    description_df["tokens"].apply(remove_stop_words)
)

# Apply to retrieval queries
queries_df["filtered_tokens"] = (
    queries_df["tokens"].apply(remove_stop_words)
)

print("Original tokens:")
print(description_df.loc[0, "tokens"])

print("\nTokens after selective stop-word removal:")
print(description_df.loc[0, "filtered_tokens"])

print("\nQuery tokens after selective stop-word removal:")
print(queries_df.loc[0, "filtered_tokens"])

Original tokens:
['a', 'rice', 'dish', 'containing', 'chicken', 'breast', 'white', 'rice', 'grape', 'tomatoes', 'and', 'yellow', 'onions']

Tokens after selective stop-word removal:
['rice', 'dish', 'containing', 'chicken', 'breast', 'white', 'rice', 'grape', 'tomatoes', 'yellow', 'onions']

Query tokens after selective stop-word removal:
['chicken', 'breast', 'accompanied', 'white', 'rice', 'tomato', 'onion']


In [12]:
# --------------------------------------------------
# Step 2.1.5 - Install NLTK
# --------------------------------------------------

import sys

!{sys.executable} -m pip install nltk

In [13]:
import nltk

nltk.download("wordnet")

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\dell\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [14]:
# --------------------------------------------------
# Step 2.1.5 - Test WordNet lemmatization
# --------------------------------------------------

from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

test_words = [
    "tomatoes",
    "onions",
    "potatoes",
    "eggs",
    "dishes",
    "chicken",
    "rice"
]

for word in test_words:
    print(f"{word:12} -> {lemmatizer.lemmatize(word, pos='n')}")

tomatoes     -> tomato
onions       -> onion
potatoes     -> potato
eggs         -> egg
dishes       -> dish
chicken      -> chicken
rice         -> rice


In [15]:
# --------------------------------------------------
# Step 2.1.5 - Apply lemmatization
# --------------------------------------------------

def lemmatize_tokens(tokens):
    """
    Apply English noun lemmatization to a token list.
    """
    return [
        lemmatizer.lemmatize(token, pos="n")
        for token in tokens
    ]


# Apply to reference descriptions
description_df["lemmatized_tokens"] = (
    description_df["filtered_tokens"].apply(lemmatize_tokens)
)

# Apply to retrieval queries
queries_df["lemmatized_tokens"] = (
    queries_df["filtered_tokens"].apply(lemmatize_tokens)
)


# Display representative examples
print("REFERENCE DESCRIPTION")
print("Before:", description_df.loc[0, "filtered_tokens"])
print("After: ", description_df.loc[0, "lemmatized_tokens"])

print("\nRETRIEVAL QUERY")
print("Before:", queries_df.loc[0, "filtered_tokens"])
print("After: ", queries_df.loc[0, "lemmatized_tokens"])

REFERENCE DESCRIPTION
Before: ['rice', 'dish', 'containing', 'chicken', 'breast', 'white', 'rice', 'grape', 'tomatoes', 'yellow', 'onions']
After:  ['rice', 'dish', 'containing', 'chicken', 'breast', 'white', 'rice', 'grape', 'tomato', 'yellow', 'onion']

RETRIEVAL QUERY
Before: ['chicken', 'breast', 'accompanied', 'white', 'rice', 'tomato', 'onion']
After:  ['chicken', 'breast', 'accompanied', 'white', 'rice', 'tomato', 'onion']


In [16]:
# --------------------------------------------------
# Step 2.1.6 - Final preprocessing comparison
# --------------------------------------------------

for i in range(3):
    print("=" * 65)

    print("DISH:", description_df.loc[i, "dish_name"])

    print("\nOriginal:")
    print(description_df.loc[i, "description"])

    print("\nCleaned:")
    print(description_df.loc[i, "clean_description"])

    print("\nFinal preprocessed text:")
    print(" ".join(description_df.loc[i, "lemmatized_tokens"]))


print("\n\nRETRIEVAL QUERY EXAMPLE")
print("=" * 65)

print("Original:")
print(queries_df.loc[0, "query_text"])

print("\nFinal preprocessed text:")
print(" ".join(queries_df.loc[0, "lemmatized_tokens"]))

DISH: Chicken with Rice

Original:
A rice dish containing chicken breast, white rice, grape tomatoes and yellow onions.

Cleaned:
a rice dish containing chicken breast white rice grape tomatoes and yellow onions

Final preprocessed text:
rice dish containing chicken breast white rice grape tomato yellow onion
DISH: Beef with Rice

Original:
A rice dish made with ground beef, white rice, grape tomatoes and yellow onions.

Cleaned:
a rice dish made with ground beef white rice grape tomatoes and yellow onions

Final preprocessed text:
rice dish made ground beef white rice grape tomato yellow onion
DISH: Salmon with Rice

Original:
A rice-based meal containing Atlantic salmon, white rice, grape tomatoes and yellow onions.

Cleaned:
a rice based meal containing atlantic salmon white rice grape tomatoes and yellow onions

Final preprocessed text:
rice based meal containing atlantic salmon white rice grape tomato yellow onion


RETRIEVAL QUERY EXAMPLE
Original:
Chicken breast accompanied by w

In [17]:
# --------------------------------------------------
# Step 2.1.7 - Inspect food vocabulary variations
# --------------------------------------------------

reference_vocabulary = {
    token
    for tokens in description_df["lemmatized_tokens"]
    for token in tokens
}

query_vocabulary = {
    token
    for tokens in queries_df["lemmatized_tokens"]
    for token in tokens
}

print("Reference vocabulary:")
print(sorted(reference_vocabulary))

print("\nQuery vocabulary:")
print(sorted(query_vocabulary))

print("\nQuery terms not present in reference vocabulary:")
print(sorted(query_vocabulary - reference_vocabulary))

Reference vocabulary:
['atlantic', 'based', 'beef', 'bowl', 'breast', 'chicken', 'combining', 'containing', 'dish', 'egg', 'grape', 'ground', 'made', 'meal', 'mixed', 'omelette', 'onion', 'pasta', 'potato', 'rice', 'russet', 'salmon', 'served', 'spaghetti', 'tomato', 'vegetable', 'white', 'whole', 'yellow']

Query vocabulary:
['accompanied', 'alongside', 'amount', 'atlantic', 'beef', 'bowl', 'breast', 'chicken', 'combined', 'combining', 'containing', 'dish', 'egg', 'fish', 'fresh', 'generous', 'grape', 'ground', 'meal', 'meat', 'minced', 'omelette', 'onion', 'pasta', 'potato', 'prepared', 'rice', 'russet', 'salmon', 'served', 'spaghetti', 'tomato', 'vegetable', 'white', 'whole', 'yellow']

Query terms not present in reference vocabulary:
['accompanied', 'alongside', 'amount', 'combined', 'fish', 'fresh', 'generous', 'meat', 'minced', 'prepared']


In [18]:
# --------------------------------------------------
# Step 2.1.8 - Final preprocessing validation
# --------------------------------------------------

# Check that all documents have tokens
assert description_df["lemmatized_tokens"].notna().all()
assert queries_df["lemmatized_tokens"].notna().all()

# Check that no document has an empty token list
assert description_df["lemmatized_tokens"].apply(len).gt(0).all()
assert queries_df["lemmatized_tokens"].apply(len).gt(0).all()

# Check that stop words were removed
for tokens in description_df["lemmatized_tokens"]:
    assert not set(tokens).intersection(custom_stop_words)

for tokens in queries_df["lemmatized_tokens"]:
    assert not set(tokens).intersection(custom_stop_words)

# Check that all tokens are lowercase
for tokens in description_df["lemmatized_tokens"]:
    assert all(token == token.lower() for token in tokens)

for tokens in queries_df["lemmatized_tokens"]:
    assert all(token == token.lower() for token in tokens)

# Calculate statistics
reference_token_count = sum(
    len(tokens) for tokens in description_df["lemmatized_tokens"]
)

query_token_count = sum(
    len(tokens) for tokens in queries_df["lemmatized_tokens"]
)

print("=" * 55)
print("FINAL PREPROCESSING VALIDATION")
print("=" * 55)

print("Processed reference descriptions:", len(description_df))
print("Processed retrieval queries:", len(queries_df))
print("Total reference tokens:", reference_token_count)
print("Total query tokens:", query_token_count)

print("\nAll preprocessing validation checks passed!")

FINAL PREPROCESSING VALIDATION
Processed reference descriptions: 10
Processed retrieval queries: 20
Total reference tokens: 98
Total query tokens: 118

All preprocessing validation checks passed!


### 2.1.9 Spelling Variation and Verb Normalization

The vocabulary inspection (2.1.7) revealed query terms absent from the
reference vocabulary. Two of these are handled at the preprocessing level:

- **Verb inflections** (combined/combining, served, prepared, made, containing)
  were not merged because lemmatization used noun mode only. A second
  lemmatization pass with verb mode is added. "ground" is protected, since
  WordNet would turn it into "grind" (the verb), which would corrupt the
  food term "ground beef".
- **Lexical variation** ("minced" vs "ground" for beef) is resolved with a small
  variation dictionary.

A few filler words ("accompanied", "alongside", "generous", "amount") carry no
information for distinguishing dishes and are removed.

Semantic relations such as fish -> salmon or meat -> beef are NOT handled here:
they are not spelling variants, and are left as a known limitation of
count-based representations (to be compared with embeddings in 2.2).

In [19]:
# --------------------------------------------------
# Step 2.1.9 - Spelling variation and verb normalization
# --------------------------------------------------

# 1. Lexical variation map (applied BEFORE lemmatization)
variation_map = {
    "minced": "ground",
}

# 2. Words that must not be verb-lemmatized (food terms)
protected_words = {"ground"}   # WordNet verb lemma of "ground" is "grind"

# 3. Filler words with no discriminative value (after lemmatization)
filler_words = {"accompany", "alongside", "generous", "amount"}


def normalize_variations(tokens):
    return [variation_map.get(token, token) for token in tokens]


def lemmatize_verbs(tokens):
    return [
        token if token in protected_words
        else lemmatizer.lemmatize(token, pos="v")
        for token in tokens
    ]


def remove_fillers(tokens):
    return [token for token in tokens if token not in filler_words]


def final_pipeline(tokens):
    """
    Starting from stop-word-filtered tokens:
    variation map -> noun lemma -> verb lemma -> filler removal
    """
    tokens = normalize_variations(tokens)
    tokens = lemmatize_tokens(tokens)    # noun lemmatization (2.1.5)
    tokens = lemmatize_verbs(tokens)
    tokens = remove_fillers(tokens)
    return tokens


description_df["final_tokens"] = description_df["filtered_tokens"].apply(final_pipeline)
queries_df["final_tokens"] = queries_df["filtered_tokens"].apply(final_pipeline)

# Joined strings (convenient for TfidfVectorizer / CountVectorizer in 2.2)
description_df["final_text"] = description_df["final_tokens"].apply(" ".join)
queries_df["final_text"] = queries_df["final_tokens"].apply(" ".join)

# --- Effect on representative examples ---
print("REFERENCE (DISH003)")
print("Before:", description_df.loc[2, "lemmatized_tokens"])
print("After: ", description_df.loc[2, "final_tokens"])

for i in [0, 3, 10]:
    print(f"\nQUERY {queries_df.loc[i, 'query_id']}")
    print("Original:", queries_df.loc[i, "query_text"])
    print("Before:  ", queries_df.loc[i, "lemmatized_tokens"])
    print("After:   ", queries_df.loc[i, "final_tokens"])

# --- Vocabulary mismatch: before vs after ---
ref_vocab = {t for toks in description_df["final_tokens"] for t in toks}
qry_vocab = {t for toks in queries_df["final_tokens"] for t in toks}

old_ref = {t for toks in description_df["lemmatized_tokens"] for t in toks}
old_qry = {t for toks in queries_df["lemmatized_tokens"] for t in toks}

print("\nQuery terms missing from reference vocabulary")
print("Before:", sorted(old_qry - old_ref))
print("After: ", sorted(qry_vocab - ref_vocab))
print(f"\nUnmatched query terms: {len(old_qry - old_ref)} -> {len(qry_vocab - ref_vocab)}")

REFERENCE (DISH003)
Before: ['rice', 'based', 'meal', 'containing', 'atlantic', 'salmon', 'white', 'rice', 'grape', 'tomato', 'yellow', 'onion']
After:  ['rice', 'base', 'meal', 'contain', 'atlantic', 'salmon', 'white', 'rice', 'grape', 'tomato', 'yellow', 'onion']

QUERY Q001
Original: Chicken breast accompanied by white rice with tomato and onion.
Before:   ['chicken', 'breast', 'accompanied', 'white', 'rice', 'tomato', 'onion']
After:    ['chicken', 'breast', 'white', 'rice', 'tomato', 'onion']

QUERY Q004
Original: White rice combined with minced beef and fresh tomatoes.
Before:   ['white', 'rice', 'combined', 'minced', 'beef', 'fresh', 'tomato']
After:    ['white', 'rice', 'combine', 'ground', 'beef', 'fresh', 'tomato']

QUERY Q011
Original: Spaghetti with minced beef and tomato.
Before:   ['spaghetti', 'minced', 'beef', 'tomato']
After:    ['spaghetti', 'ground', 'beef', 'tomato']

Query terms missing from reference vocabulary
Before: ['accompanied', 'alongside', 'amount', 'combi

In [20]:
# --------------------------------------------------
# Step 2.1.10 - Final preprocessing validation
# --------------------------------------------------

assert description_df["final_tokens"].apply(len).gt(0).all()
assert queries_df["final_tokens"].apply(len).gt(0).all()

# Protected term preserved, variants merged
all_final = {t for toks in description_df["final_tokens"] for t in toks} | \
            {t for toks in queries_df["final_tokens"] for t in toks}

assert "ground" in all_final and "grind" not in all_final
assert "minced" not in all_final
assert not (all_final & filler_words)

print("Reference tokens:", sum(len(t) for t in description_df["final_tokens"]))
print("Query tokens:    ", sum(len(t) for t in queries_df["final_tokens"]))
print("Final vocabulary size:", len(all_final))
print("\nPreprocessing pipeline complete. Use 'final_text' / 'final_tokens' for 2.2.")

Reference tokens: 98
Query tokens:     112
Final vocabulary size: 33

Preprocessing pipeline complete. Use 'final_text' / 'final_tokens' for 2.2.
